In [2]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

FINMIND_TOKEN = os.getenv("FINMIND_TOKEN", "").strip()

url = "https://api.finmindtrade.com/api/v4/taiwan_futures_snapshot"

headers = {
    "Authorization": f"Bearer {FINMIND_TOKEN}"
}

params = {
    "data_id": "TMF"   # 微型臺指期貨
}

resp = requests.get(url, headers=headers, params=params, timeout=20)

print("HTTP status:", resp.status_code)

data = resp.json()

print("\nRaw response:")
print(data)

print("\nResponse keys:")
print(data.keys())

rows = data.get("data", [])

print("\nData type:", type(rows))
print("Data length:", len(rows) if isinstance(rows, list) else "not list")

df = pd.DataFrame(rows)

print("\nDF shape:", df.shape)
print("DF columns:")
print(df.columns.tolist())

if df.empty:
    print("\nNo data returned. Most likely this endpoint requires sponsor access or your token is invalid.")
else:
    print("\nPreview:")
    print(df.head())

    wanted_cols = ["futures_id", "close", "buy_price", "sell_price", "date"]
    existing_cols = [col for col in wanted_cols if col in df.columns]

    print("\nAvailable selected columns:")
    print(df[existing_cols].head())

    if "futures_id" in df.columns:
        print("\nAll TMF contract IDs:")
        print(df["futures_id"].unique())

        # Later we filter May contract after confirming exact code
        may_contracts = df[df["futures_id"].astype(str).str.contains("TMF", na=False)]
        print("\nTMF rows:")
        print(may_contracts[existing_cols])

HTTP status: 400

Raw response:
{'msg': 'Your level is register. Please update your user level. Detail information:https://finmindtrade.com/analysis/#/Sponsor/sponsor', 'status': 400}

Response keys:
dict_keys(['msg', 'status'])

Data type: <class 'list'>
Data length: 0

DF shape: (0, 0)
DF columns:
[]

No data returned. Most likely this endpoint requires sponsor access or your token is invalid.


In [ ]:
import os
import time
import requests
import pandas as pd
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
from dotenv import load_dotenv

load_dotenv()

FINMIND_TOKEN = os.getenv("FINMIND_TOKEN", "").strip()
N8N_WEBHOOK_URL = os.getenv("N8N_WEBHOOK_URL", "").strip()
ALERT_SECRET = os.getenv("ALERT_SECRET", "").strip()

FINMIND_URL = "https://api.finmindtrade.com/api/v4/data"
DATASET = "TaiwanVariousIndicators5Seconds"

# Change this to your test threshold
TAIEX_ALERT_THRESHOLD = 4000000


def fetch_taiex_for_date(date_str: str) -> pd.DataFrame:
    headers = {}

    # FinMind supports token-based authentication.
    # If token is empty, it will try without token, but quota may be lower.
    if FINMIND_TOKEN:
        headers["Authorization"] = f"Bearer {FINMIND_TOKEN}"

    params = {
        "dataset": DATASET,
        "start_date": date_str,
        "end_date": date_str,
    }

    response = requests.get(
        FINMIND_URL,
        headers=headers,
        params=params,
        timeout=20
    )

    print("HTTP status:", response.status_code)

    response.raise_for_status()
    result = response.json()

    if "data" not in result:
        print("Unexpected response:")
        print(result)
        return pd.DataFrame()

    return pd.DataFrame(result["data"])


def get_latest_taiex(max_days_back: int = 7):
    """
    Try today first.
    If market is closed or no data is returned, check previous days.
    """
    taipei_now = datetime.now(ZoneInfo("Asia/Taipei"))

    for i in range(max_days_back):
        target_date = (taipei_now - timedelta(days=i)).strftime("%Y-%m-%d")
        print(f"\nChecking date: {target_date}")

        df = fetch_taiex_for_date(target_date)

        if df.empty:
            print("No data for this date.")
            continue

        if "date" not in df.columns or "TAIEX" not in df.columns:
            print("Unexpected columns:", list(df.columns))
            print(df.head())
            continue

        df["date"] = pd.to_datetime(df["date"])
        df["TAIEX"] = pd.to_numeric(df["TAIEX"], errors="coerce")
        df = df.dropna(subset=["TAIEX"]).sort_values("date")

        if df.empty:
            print("No valid TAIEX rows.")
            continue

        latest = df.iloc[-1]

        return {
            "date": latest["date"],
            "taiex": float(latest["TAIEX"]),
            "source_date": target_date,
            "row_count": len(df),
        }

    return None


def send_n8n_alert(latest_data):
    if not N8N_WEBHOOK_URL:
        print("N8N_WEBHOOK_URL is empty. Skipping webhook test.")
        return

    payload = {
        "alert_name": "TAIEX threshold test",
        "symbol": "TAIEX",
        "value": latest_data["taiex"],
        "threshold": TAIEX_ALERT_THRESHOLD,
        "data_time": str(latest_data["date"]),
        "message": f"TAIEX is {latest_data['taiex']}, threshold is {TAIEX_ALERT_THRESHOLD}",
    }

    headers = {}

    if ALERT_SECRET:
        headers["X-Alert-Secret"] = ALERT_SECRET

    response = requests.post(
        N8N_WEBHOOK_URL,
        json=payload,
        headers=headers,
        timeout=20
    )

    print("n8n webhook status:", response.status_code)
    print("n8n response:", response.text[:500])


def main():
    latest = get_latest_taiex()

    if latest is None:
        print("\nCould not get TAIEX data from FinMind.")
        return

    print("\nLatest TAIEX data:")
    print("Data time:", latest["date"])
    print("TAIEX:", latest["taiex"])
    print("Rows returned:", latest["row_count"])

    if latest["taiex"] >= TAIEX_ALERT_THRESHOLD:
        print(f"\nALERT HIT: TAIEX >= {TAIEX_ALERT_THRESHOLD}")
        send_n8n_alert(latest)
    else:9
        print(f"\nNo alert: TAIEX < {TAIEX_ALERT_THRESHOLD}")


if __name__ == "__main__":
    main()


Checking date: 2026-04-27
HTTP status: 200

Latest TAIEX data:
Data time: 2026-04-27 13:30:00
TAIEX: 39616.63
Rows returned: 3241

No alert: TAIEX < 4000000


In [3]:
pip install lxml

Note: you may need to restart the kernel to use updated packages.


In [4]:
import requests
import pandas as pd
from io import StringIO

URL = "https://www.taifex.com.tw/cht/3/futDailyMarketExcel"

params = {
    "commodity_id": "TMF"   # 微型臺指期貨
}

headers = {
    "User-Agent": "Mozilla/5.0"
}

resp = requests.get(URL, params=params, headers=headers, timeout=20)
print("HTTP status:", resp.status_code)

resp.encoding = "utf-8"

# Read all HTML tables from TAIFEX page
tables = pd.read_html(StringIO(resp.text))

print("Number of tables found:", len(tables))

for i, table in enumerate(tables):
    print(f"\n--- Table {i} ---")
    print(table.head())
    print("Columns:", table.columns.tolist())

# Usually the useful futures table is one of the bigger tables
df = max(tables, key=len)

# Flatten columns if pandas reads multi-level headers
if isinstance(df.columns, pd.MultiIndex):
    df.columns = [
        "_".join([str(x) for x in col if str(x) != "nan"]).strip()
        for col in df.columns
    ]
else:
    df.columns = [str(col).strip() for col in df.columns]

print("\nSelected table shape:", df.shape)
print("Selected columns:")
print(df.columns.tolist())

print("\nPreview:")
print(df.head(20))

# Convert all values to string for easier searching
df_str = df.astype(str)

# Find rows containing TMF and 202605
target_rows = df[
    df_str.apply(lambda row: row.str.contains("TMF", na=False).any(), axis=1)
    & df_str.apply(lambda row: row.str.contains("202605", na=False).any(), axis=1)
]

print("\nTarget: 微台指期 2026/05")
print(target_rows)

HTTP status: 200
Number of tables found: 3

--- Table 0 ---
    契約  到期 月份 (週別)      開盤價      最高價      最低價   最後 成交價   漲跌價     漲跌%  \
0  TMF    202605.0  39990.0  40445.0  39838.0  39900.0  ▲678  ▲1.73%   
1  TMF    202606.0  40085.0  40532.0  39939.0  39980.0  ▲670  ▲1.70%   
2  TMF    202607.0  39999.0  40480.0  39900.0  39940.0  ▲682  ▲1.74%   
3  TMF    202609.0  40150.0  40580.0  40000.0  40042.0  ▲689  ▲1.75%   
4  TMF    202612.0  40358.0  40905.0  40343.0  40450.0  ▲763  ▲1.92%   

   *盤後交易時段成交量  *一般交易時段成交量  *合計成交量      結算價  *未沖銷契約量   最後最佳買價   最後最佳賣價  \
0      220363      181953  402316  39921.0    39901  39889.0  39890.0   
1       12096        9778   21874  39997.0     8102  39979.0  39990.0   
2        1400        1431    2831  39949.0     1384  39939.0  39950.0   
3        1132         735    1867  40047.0     2928  40050.0  40051.0   
4         405         347     752  40390.0     1433  40368.0  40402.0   

     歷史最高價    歷史最低價  
0  40445.0  31308.0  
1  40532.0  20700.0  
2 